In [61]:
import gurobipy as gp
from gurobipy import GRB

3.8

### Implementation using Big-M reformulation of Complementarity Constraint

In [72]:
model_bigM = gp.Model("milp_sos1")

M_slack = [10000, 10000, 10000, 10000]
M_lamda = [10000, 10000, 10000, 10000]

x = model_bigM.addVar(lb=-GRB.INFINITY, name="x")
y = model_bigM.addVar(lb=-GRB.INFINITY, name="y")
z = model_bigM.addVars(4, vtype=GRB.BINARY, name="z")

lamda = model_bigM.addVars(4, lb=0, name="lamda")
slack = model_bigM.addVars(4, lb=0, name="slack")

model_bigM.addConstr(x - 2*y >= -8, "upper")
model_bigM.addConstr(slack[0] == - 7 + x + y, "slack1")
model_bigM.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
model_bigM.addConstr(slack[2] == 4 - x + 2*y, "slack3")
model_bigM.addConstr(slack[3] == 13 - x - y, "slack4")

model_bigM.addConstr(lamda[0] + 5*lamda[1] + 2*lamda[2] -lamda[3] == 1, "df")


model_bigM.addConstrs((slack[i] <= M_slack[i] * (1 - z[i]) for i in range(4)), "bimM_slack")
model_bigM.addConstrs((lamda[i] <= M_lamda[i] * z[i] for i in range(4)), "bigM_dual")

model_bigM.setObjective(x - y, GRB.MINIMIZE)
model_bigM.params.OutputFlag=0
model_bigM.optimize()

In [74]:
if model_bigM.Status == GRB.OPTIMAL:
    print(f"Objective = {model_bigM.ObjVal:.2f}")
    print(f"x = {x.X:.2f}")
    print(f"y = {y.X:.2f}")

    for i in range(4):
        print(
            f"i={i+1}: "
            f"z={z[i].X:.0f}, "
            f"lambda={lamda[i].X:.2f}, "
            f"slack={slack[i].X:.2f}, "
            f"product={lamda[i].X * slack[i].X:.2e}"
        )

Objective = -3.00
x = 2.00
y = 5.00
i=1: z=1, lambda=1.00, slack=0.00, product=0.00e+00
i=2: z=-0, lambda=0.00, slack=21.00, product=0.00e+00
i=3: z=-0, lambda=0.00, slack=12.00, product=0.00e+00
i=4: z=-0, lambda=0.00, slack=6.00, product=0.00e+00


Is the value of Big-M correct? We chose arbitrarily a very big number. It might remove some original solutions or it might be far away from being tight. In the next step, we try to get tighter values of Big Ms.

In [75]:
# Finding minimum x
model = gp.Model("model")

x = model.addVar(lb=-GRB.INFINITY, name="x")
y = model.addVar(lb=-GRB.INFINITY, name="y")
slack = model.addVars(4, lb=0, name="slack")

model.addConstr(x - 2*y >= -8, "upper")
model.addConstr(slack[0] == - 7 + x + y, "slack1")
model.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
model.addConstr(slack[2] == 4 - x + 2*y, "slack3")
model.addConstr(slack[3] == 13 - x - y, "slack4")

model.Params.OutputFlag = 0
model.setObjective(x, GRB.MINIMIZE)
model.optimize()
print(model.ObjVal)

2.0


Finding M Values for dual variables of lower level problems

In [76]:
#  Finding maximum x
model.Params.OutputFlag = 0
model.setObjective(x, GRB.MAXIMIZE)
model.optimize()
print(model.ObjVal)

10.0


$ 2 \leq x \leq 10$

In [77]:
#  Finding minimum y
model.Params.OutputFlag = 0
model.setObjective(y, GRB.MINIMIZE)
model.optimize()
print(model.ObjVal)

1.5


In [78]:
#  Finding maximum y
model.Params.OutputFlag = 0
model.setObjective(y, GRB.MAXIMIZE)
model.optimize()
print(model.ObjVal)
model.reset()


7.0


$ 1.5 \leq x \leq 7$

In [79]:
# Finding max slack
model = gp.Model("model")
x = model.addVar(lb=2, ub=10, name="x")
y = model.addVar(lb=1.5, ub=7, name="y")
slack = model.addVars(4, lb=0, name="slack")

model.addConstr(x - 2*y >= -8, "upper")
model.addConstr(slack[0] == - 7 + x + y, "slack1")
model.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
model.addConstr(slack[2] == 4 - x + 2*y, "slack3")
model.addConstr(slack[3] == 13 - x - y, "slack4")
model.Params.OutputFlag = 0
for i in range(4):
    model.setObjective(slack[i], GRB.MAXIMIZE)
    model.optimize()
    print(f"M_slack[{i}] = {model.ObjVal}")

M_slack[0] = 6.0
M_slack[1] = 27.0
M_slack[2] = 12.0
M_slack[3] = 6.0


Finding bounds for variables solving LPs to determine tight Big-M values for Complementary constraints slack. Using similar approach to above gives us unbounded LP when we try to find the bounds for dual variables of the lower level problems. Finding bounds for lamda is not trivial. So we test with different values and see if that changes the solution. We test with 10, 100, 1000, 10000, 1e5, 1e6, 1e7

In [80]:
model.reset()
for a in [1e-1, 1, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8]:
    model_bigM = gp.Model("milp_sos1")
    M_slack = [6, 27, 12, 6]
    M_lamda = [a, a, a, a]

    x = model_bigM.addVar(lb=-GRB.INFINITY, name="x")
    y = model_bigM.addVar(lb=-GRB.INFINITY, name="y")
    z = model_bigM.addVars(4, vtype=GRB.BINARY, name="z")

    lamda = model_bigM.addVars(4, lb=0, name="lamda")
    slack = model_bigM.addVars(4, lb=0, name="slack")

    model_bigM.addConstr(x - 2*y >= -8, "upper")
    model_bigM.addConstr(slack[0] == - 7 + x + y, "slack1")
    model_bigM.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
    model_bigM.addConstr(slack[2] == 4 - x + 2*y, "slack3")
    model_bigM.addConstr(slack[3] == 13 - x - y, "slack4")

    model_bigM.addConstr(lamda[0] + 5*lamda[1] + 2*lamda[2] -lamda[3] == 1, "df")


    model_bigM.addConstrs((slack[i] <= M_slack[i] * (1 - z[i]) for i in range(4)), "bimM_slack")
    model_bigM.addConstrs((lamda[i] <= M_lamda[i] * z[i] for i in range(4)), "bigM_dual")

    model_bigM.setObjective(x - y, GRB.MINIMIZE)
    model_bigM.params.OutputFlag=0
    model_bigM.optimize()

    if model_bigM.Status == GRB.OPTIMAL:
        print(f"With M_lamda={a}, Optimal solution is{model_bigM.ObjVal}")
    else: 
        print(model_bigM.Status)
    model.reset()

4
With M_lamda=1, Optimal solution is-3.0
With M_lamda=10.0, Optimal solution is-3.0
With M_lamda=100.0, Optimal solution is-3.0
With M_lamda=1000.0, Optimal solution is-3.0
With M_lamda=10000.0, Optimal solution is-3.0
With M_lamda=100000.0, Optimal solution is-3.0
With M_lamda=1000000.0, Optimal solution is-3.0
With M_lamda=10000000.0, Optimal solution is-3.0
With M_lamda=100000000.0, Optimal solution is-3.0


With small M, we got infeasible or unbounded model. For larger M, the objective values is the same. We cannot claim yet that this is correct solution. Therefore, we use strong duality and see if we get the same result.

### Implementation using Strong duality

In [81]:
model_SD = gp.Model("milp_sos1")

x = model_SD.addVar(lb=-GRB.INFINITY, name="x")
y = model_SD.addVar(lb=-GRB.INFINITY, name="y")

lamda = model_SD.addVars(4, lb=0, name="lamda")
slack = model_SD.addVars(4, lb=0, name="slack")

model_SD.addConstr(x - 2*y >= -8, "upper")
model_SD.addConstr(slack[0] == - 7 + x + y, "slack1")
model_SD.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
model_SD.addConstr(slack[2] == 4 - x + 2*y, "slack3")
model_SD.addConstr(slack[3] == 13 - x - y, "slack4")

model_SD.addConstr(lamda[0] + 5*lamda[1] + 2*lamda[2] -lamda[3] == 1, "df")

model_SD.addConstr(y == (7-x) * lamda[0] + (x+2) * lamda[1] + (x-4) * lamda[2] + (x-13) * lamda[3], "strong_duality")
model_SD.setObjective(x - y, GRB.MINIMIZE)

model_SD.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 10 columns and 18 nonzeros (Min)
Model fingerprint: 0xb6222eae
Model has 2 linear objective coefficients
Model has 1 quadratic constraint
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+01]

Presolve removed 2 rows and 4 columns

Continuous model is non-convex -- solving as a MIP


Presolve removed 0 rows and 4 columns
Presolve time: 0.00s
Presolved: 28 rows, 12 columns, 84 nonzeros
Presolved model has 5 bilinear constraint(s)
         in product terms.
         Presolve was not able to compute smaller bounds for these variables.
         Cons

Using strong duality based reformulation, we get the same solution. Therefore, our BigM values related to the dual variables of lower level problems was indeed good as long as it was greater than or equal to 1. Using SD based approach, note that we have bilinear terms that makes the problem non linear. This is one possible disadvantage of SD based reformulation, however, Gurobi can handles these terms. So, this is more prefearble to BigM as there is no need to solve Fourier Motzkin Elimination or Auxillay LPs (as we did) to finds the value of Ms. Also, finding valus of M related to dual variables is not straightforward. 

5.13

### Implementation defining SOS1 Constraints

In [83]:
model_SOS1 = gp.Model("milp_sos1")

x = model_SOS1.addVar(lb=-GRB.INFINITY, name="x")
y = model_SOS1.addVar(lb=-GRB.INFINITY, name="y")

lamda = model_SOS1.addVars(4, lb=0, name="lamda")
slack = model_SOS1.addVars(4, lb=0, name="slack")

model_SOS1.addConstr(x - 2*y >= -8, "upper")
model_SOS1.addConstr(slack[0] == - 7 + x + y, "slack1")
model_SOS1.addConstr(slack[1] == - 2 - x + 5*y, "slack2")
model_SOS1.addConstr(slack[2] == 4 - x + 2*y, "slack3")
model_SOS1.addConstr(slack[3] == 13 - x - y, "slack4")

model_SOS1.addConstr(lamda[0] + 5*lamda[1] + 2*lamda[2] -lamda[3] == 1, "df")

for i in range(4):
    model_SOS1.addSOS(GRB.SOS_TYPE1, [slack[i], lamda[i]])
model_SOS1.setObjective(x - y, GRB.MINIMIZE)

model_SOS1.optimize()


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26300.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 10 columns and 18 nonzeros (Min)
Model fingerprint: 0x09cf2936
Model has 2 linear objective coefficients
Model has 4 SOS constraints
Variable types: 10 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+01]

Presolve removed 3 rows and 2 columns
Presolve time: 0.00s
Presolved: 3 rows, 8 columns, 9 nonzeros
Presolved model has 4 SOS constraint(s)
Variable types: 8 continuous, 0 integer (0 binary)

Root relaxation: objective -3.000000e+00, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth

Using SOS1 type constraints defintion, we again reach to the same solutions. Note that, this SOS1 based formulation is a MILP.

Using all of these three methods, we reach the optimal solution. 